# Relevance-atom judge — Phase 1 pilot

**Objective.** Drive the `relevance_judge` package one gate at a time and inspect the
results as live objects. The judge asks a single question per (query, doc) — *"is this
document relevant to this query?"* — and never sees route names, lists, or scores. New
judged pairs extend the qrels; route scores recompute arithmetically at scoring time.

**Done** = the accuracy gate passes on positive precision, the sub-1.0-tie pilot has judged
its ~396 above-gold pairs, and `PilotScorer` reports the qrels-hole rate + per-route
discovered-relevance.

Plan: `~/.claude/plans/can-you-use-the-shimmering-hare.md` · doc:
`docs/research/relevance-judge-recovery.md` · CLI twin: `src/scripts/run_relevance_judge.py`.

In [ ]:
# Setup: imports, config, and a spend guard (nothing here calls an LLM)
from __future__ import annotations

import pandas as pd

from augmentation.engine import Budget
from relevance_judge import (
    JudgeQueue,
    JudgeRunLog,
    PilotScorer,
    RelevanceJudge,
    RelevanceJudgeConfig,
    Sources,
    ValidationHarness,
)

pd.set_option("display.max_colwidth", 90)

SEED = 0
SPEND = True          # flip to True to make real LLM calls (needs OPENROUTER_API_KEY + gpt-5.6-luna)
MAX_SPEND_USD = 5.0    # hard ceiling enforced per-call by Budget

# Tied ROWS to build a work list from (None = all 1,917 -> ~30.8K pairs, ~$13).
# The residual is lane-ORDERED, so a limit takes one lane's rows, not a sample
# across lanes: 300 -> 4,816 pairs, all `clerc`, ~$2.17. clerc is at the dear
# end (1,448-char median query), so this over-states cost/pair for the mix —
# a conservative probe, but it measures one lane, not six.
TIE_ROW_LIMIT = 300

# Reasoning is the precision/latency dial. 'none' = ~1.2s/call but precision
# fell 0.955 -> 0.905 (the freshstack code lanes carry 50 of 56 false positives);
# None omits the param entirely = full reasoning, ~60s/call. 'low'/'medium'/'high'
# are the untested middle. Re-run §1.1 after changing it — the gate is per-config.
REASONING_EFFORT = "none"

config = RelevanceJudgeConfig(reasoning_effort=REASONING_EFFORT)


def budget() -> Budget:
    return Budget(
        MAX_SPEND_USD,
        usd_per_mtok_in=config.engine.usd_per_mtok_in,
        usd_per_mtok_out=config.engine.usd_per_mtok_out,
    )


config.engine.model, config.artifacts

## The gate ladder

Each stage gates the next; the free ones cannot be invalidated by the open SPEC decision.

| stage | spend | what it decides |
|---|---|---|
| §1.0 precondition audit | free | how many residual rows can be rescored at all |
| §1.1 accuracy gate | ~$1 | **positive precision** — is the judge safe to trust as gold? |
| §1.2 sub-1.0-tie pilot | ~$0.12 | judge above-gold docs; break ties arithmetically |
| score | free | qrels-hole rate · per-route discovered-relevance · tie-conversion |

Merge rule: judged atoms carry `source="llm"`; `QrelStore` precedence means a human
judgment always wins a conflict, so human truth stays authoritative.

## §1.0 — Precondition audit (free)

Reproduces the arch5k residual: `all_tied` ∪ `absent@500`, and how many have persisted `route_rankings` to rescore against.

In [ ]:
queue = JudgeQueue(config)
audit = queue.precondition_audit()
pd.Series(audit, name="residual population")

## §1.1 — Accuracy gate (spends ~$1 when `SPEND=True`)

The **hard gate is precision + a recall floor**, not agreement: a judged atom becomes gold, so a
false positive injects wrong truth (the dangerous error); a conservative judge that answers
strictly is *safe* for gold injection, so overall/per-lane agreement are reported as diagnostics,
not enforced. Stages are inspectable — `sample()` → `judge_rows()` → `score()` — a per-lane
sampling table, a live progress bar, per-pair predictions in `validation_predictions.parquet`,
a confusion breakdown, and a false-positive audit.

**Two caveats this surfaces.** (1) Precision is only measurable where humans judged *irrelevant*
docs; residual cover lanes are positive-only, so the gate borrows negatives from graded lanes
(non-human/synthetic lanes like `augmentation` are excluded automatically). (2) Some datasets
grade *topical/partial* matches relevant while the judge answers strict *"does this answer the
query"* — so low recall there is a definition gap, not a judge failure. Always audit the false
positives (last cell): a grade-0 doc the judge calls relevant may be a genuine qrels-hole.

In [ ]:
# The judge's instruction — the precision lever. To iterate: edit
# relevance_judge/judge.py::INSTRUCTION, restart the kernel, re-run --validate.
from relevance_judge.judge import INSTRUCTION
print(INSTRUCTION)

In [ ]:
# Lane selection (free): where the judge is applied vs where precision is measurable
harness = ValidationHarness(config)
residual_lanes = sorted(queue.residual()["dataset"].unique())
negative_lanes = Sources(config).lanes_with_negatives()
val_lanes = sorted(set(residual_lanes) | set(negative_lanes))

print("residual lanes (judge applied here):", residual_lanes)
print("lanes WITH judged negatives (precision measurable):", negative_lanes)
print("residual lanes WITHOUT negatives (recall-only):",
      sorted(set(residual_lanes) - set(negative_lanes)))

## §1.1a — Per-lane cards from `CardGenerator` (SPEC d71) — installed BEFORE the gate

The pilot §1.2 below banks permanent gold using a **per-lane card + universal
INSTRUCTION**. So the validation gate must measure the SAME instrument — a
gate that scored the universal-only judge would not certify the carded one.
Cards therefore install here, before `judge_rows()` runs.

**Architecture** (SPEC d71). `CardGenerator` writes one card per lane from a
metadata bundle (docstring + stratified queries + up-to-3
human-judged-irrelevant snippets per query). Temperature=0, cached to
`generated_cards.parquet` — a rerun compares the same instance across arms.

**Why it works.** Paired A/B on the 6 negative-bearing lanes
(`scripts/card_transfer.py`) established: safe on all 6 (Δfpr ≤ +0.01, max 1
card-caused FP on 100 negatives), matches hand-written cards on task-family
twins (dbpedia proxies `quest`: +18pp recall; crumb-clinical proxies
`crumb-legal-qa`: +11pp). Three-agent triangulation identified the residual
ceiling as **definitional, not technical** — grade-1 partial matches a
strict judge correctly rejects.

**Complementary change**: `LANES` sets `min_relevance=2` on the three lanes
with a grade-2 tier (crumb-clinical-trial, dbpedia-entity, wands). The judge
is unchanged; the harness stops counting grade-1 partial-topic matches as
positives — the definitional gap closes on metrics, not on behavior.


In [ ]:
# The generator's meta-prompt — the analogue of cell 6's INSTRUCTION for the
# judge itself. This is what luna receives as system context; the metadata
# bundle in the next cell is the user turn. To iterate: edit
# card_generator.py::_GENERATOR_INSTRUCTION, restart the kernel, delete
# generated_cards.parquet, re-run §1.1a.
from relevance_judge.card_generator import _GENERATOR_INSTRUCTION
print(_GENERATOR_INSTRUCTION)


In [ ]:
# The exact input G sees for ONE lane, so you can predict what card it will
# write. Same _lane_metadata() the generator calls internally. Swap the lane
# name below to check any of the 6 deploy lanes.
from relevance_judge.card_generator import _lane_metadata

sample_lane = "clerc"   # deploy lane — swap for finder / quest / scirgen-geo-en / crumb-legal-qa / rarb-math
print(_lane_metadata(sample_lane, Sources(config)))


In [ ]:
# Generate a card per deploy lane. Cached to generated_cards.parquet — reruns
# skip lanes with a cached card (temperature=0 makes the cache correct). To
# force a fresh draw, delete the file or pass regenerate=True.
from relevance_judge.card_generator import CardGenerator
from relevance_judge.lane_context import LaneContext

deploy_lanes = sorted(queue.residual()["dataset"].unique())
_gen = CardGenerator(config)

if SPEND:
    generated = _gen.generate_all(deploy_lanes)
else:
    # dry mode: return only what is already cached, don't call the model
    _cached = _gen._cache()
    generated = {
        row.lane: LaneContext(task=row.task, queries=row.queries,
                              gold=row.gold, judging=row.judging)
        for row in _cached.itertuples(index=False) if row.lane in deploy_lanes
    }
    print(f"cached cards for {len(generated)}/{len(deploy_lanes)} lanes "
          f"(set SPEND=True to fill the rest at ~$0.01/lane)")


In [ ]:
# Inspect each generated card — JUDGING is the safety-critical field. Read
# every row: a card that only names task/gold but forgets to identify the
# near-miss is inert; a card whose JUDGING loosens "answer" toward "topic"
# should be regenerated (delete generated_cards.parquet) before the gate runs.
if generated:
    display(pd.DataFrame([
        {"lane": lane, "task": c.task[:80], "queries": c.queries[:80],
         "gold": c.gold[:80], "judging": c.judging[:220]}
        for lane, c in sorted(generated.items())
    ]))
else:
    print("no cards to inspect — flip SPEND=True and rerun the previous cell to generate")


In [ ]:
# Install the generated cards for the rest of the notebook — the validation
# gate that runs next AND the pilot §1.2 that banks gold both read from these.
# Same instrument measured and deployed. Reversible: restart the kernel.
import relevance_judge.judge as _jmod

_hand_cards = {k for k, v in __import__(
    "relevance_judge.lane_context", fromlist=["LANE_CONTEXT"]
).LANE_CONTEXT.items() if v}
_shadowed = _hand_cards & set(generated)
if _shadowed:
    print(f"NOTE: generated cards shadow hand-written cards on: {sorted(_shadowed)}")
    print("  (the hand card in lane_context.py is left in place; runtime lookup uses `generated`)")

_jmod.context_for = lambda dataset: (
    generated[dataset].render() if dataset in generated else ""
)
print(f"installed {len(generated)} generated cards — §1.1 and §1.2 now use them")


In [ ]:
# Stage 1 — sample() is FREE: builds the eval frame and prints a per-lane pos/neg/dropped table
eval_rows = harness.sample(val_lanes, per_lane=200, seed=SEED)
eval_rows.groupby(["dataset", "human_relevant"]).size().unstack(fill_value=0)

In [ ]:
# Stage 2 — judge_rows() shows a live tqdm bar and banks validation_predictions.parquet
# every 50 rows (crash-safe). It reuses eval_rows from Stage 1 (no re-sampling) and returns
# the per-pair predictions as an inspectable frame.
if SPEND:
    preds = harness.judge_rows(eval_rows, budget=budget())
    preds.head()
else:
    preds = None
    print(f"set SPEND=True to judge {len(eval_rows)} validation pairs (tqdm bar appears here)")

In [ ]:
# Stage 3 — score() is pure (no LLM); finalize() writes the report + opens a judge run on pass
if preds is not None:
    report = harness.finalize(harness.score(preds, val_lanes))
    print("GATE:", "PASS" if report["passed"] else "FAIL", "->", config.validation_report)
    report_view = {k: v for k, v in report.items() if k != "agreement_by_lane"}
else:
    report_view = {"skipped": "run Stage 2 with SPEND=True first"}

report_view

### Re-score without spending

The judgments are banked in `validation_predictions.parquet`; scoring is pure arithmetic. So after
any gate change (thresholds, referee rules) you re-derive the verdict for **free** — no re-judging.
This is the notebook twin of `run_relevance_judge.py --validate --rescore`.

In [ ]:
# Re-score the banked predictions under the CURRENT gate logic — NO LLM spend (read-only here).
# To also open a judge run (unlocks the pilot), use the CLI: `--validate --rescore`.
if config.validation_predictions.exists():
    banked = pd.read_parquet(config.validation_predictions)
    valid = set(Sources(config).lanes_with_negatives()) | set(config.anchor_lanes)
    # drop lanes that only LOOK negative under a stale threshold (e.g. nfcorpus grade-1)
    keep = banked.groupby("dataset").filter(
        lambda g: g.name in valid or not bool((~g["human_relevant"]).any())
    )
    rescored = harness.score(keep, sorted(keep["dataset"].unique()))
    print("GATE (rescored, no spend):", "PASS" if rescored["passed"] else "FAIL")
    rescored_view = {k: rescored[k] for k in (
        "passed", "precision_relevant", "anchor_recall", "recall_relevant",
        "false_positives", "n_validation")}
else:
    rescored_view = {"skipped": "no banked predictions yet — run Stage 2 once"}
rescored_view

In [ ]:
# Audit the false positives — the whole reason precision is the gate. audit_false_positives()
# enriches each FP with the human grade + query/doc text and banks false_positives_audit.parquet.
# Hand-check each: a genuine judge error, or a qrels-hole (relevant-but-unjudged) the program exists
# to find? (A grade-0 doc the judge calls relevant may be a real hole, not a mistake.)
if config.validation_predictions.exists():
    fp_audit = harness.audit_false_positives(per_lane=10)
    print(f"{len(fp_audit)} false positives; by lane:")
    print(fp_audit["dataset"].value_counts().to_string())
    display(fp_audit[["dataset", "human_grade", "judge_reason", "query", "doc"]].head(15))
else:
    print("no predictions yet — run Stage 2 with SPEND=True")

## §1.2 — Sub-1.0-tie pilot

The work list is free to build (below) — above-gold docs only, the only docs whose relevance
can break a sub-1.0 tie. The drop accounting is printed: `no-rankings` / `gold-at-v2-rank-1`
(the l2-tie vs persisted-v2-ranking mismatch) / `no-text`.

In [27]:
pairs = queue.tie_pairs(limit=TIE_ROW_LIMIT)   # prints the drop accounting
pairs.head()

tied rows: 300; dropped 0 no-rankings, 0 identical-lists, 0 no-text; 4816 pairs to judge


,dataset,query_id,doc_id,query,doc_text,rank_spread
0,clerc,101064,23247479,excessive risk to inmate health or safety; the official must both [have been] aware of...,of medical care or inadequacy in the treatment administered. These examples fortify th...,10
1,clerc,101691,5612302,"the time and place said machine was used. Such production is proper under Rule 16(b), ...",these rights and that he did not have any questions. Based on our review of the record...,10
2,clerc,123300,11497603,accordance with a subpoena issued by an officer or agency of the United States under a...,"aff’d sub nom., Placid Oil Co. v. FPC, 483 F.2d 880 (5th Cir. 1973), aff’d sub nom., M...",10
3,clerc,123300,22053986,accordance with a subpoena issued by an officer or agency of the United States under a...,public rule making proceedings; (2) reference to the legal authority under which the r...,10
4,clerc,249058,14726006,in its entirety as an exception to the hearsay rule under Rule 803(8)(C) because the d...,"district court, having observed the witnesses testify to their different versions of e...",10


In [ ]:
# Judge the pairs — needs SPEND=True and a passed §1.1 run.
# Cost is measured off THESE pairs (no truncation happens: judge.py sends the
# full query + full doc), so it tracks the lane mix instead of a flat guess.
_ptok = (pairs["doc_text"].str.len() + pairs["query"].str.len() + 400) / 4
_est = float(
    (_ptok / 1e6 * config.engine.usd_per_mtok_in).sum()
    + len(pairs) * 35 / 1e6 * config.engine.usd_per_mtok_out
)
_banked = len(RelevanceJudge(config).judged_keys())
print(f"{len(pairs):,} pairs | {_banked:,} already banked (skipped) | "
      f"projected ~${_est:.2f} vs ceiling ${MAX_SPEND_USD:.2f}")
if _est > MAX_SPEND_USD:
    print(f"  NOTE: the ceiling stops the run early — lower TIE_ROW_LIMIT or raise MAX_SPEND_USD")

runs = JudgeRunLog(config).load()
passed = runs[runs["passed"]] if not runs.empty else runs
run_id = str(passed.iloc[-1]["judge_run_id"]) if not passed.empty else None

print(SPEND and run_id)
if SPEND and run_id:
    counts = RelevanceJudge(config).judge_pairs(pairs, run_id=run_id, budget=budget())
elif SPEND:
    counts = {"error": "no passed validation run — run §1.1 first"}
else:
    counts = {"skipped": f"set SPEND=True (after §1.1) to judge {len(pairs)} pairs (~${_est:.2f})"}

counts

## Results — what the atoms bought

Returns `{'error': 'no judged atoms yet'}` until §1.2 has run. `qrels_hole_rate` is the headline (stack-independent); `discovered_relevance_by_route` is the sparse blind-spot audit; `tie_conversion_rate` uses the persisted v2 rankings (a stated approximation for l2-defined ties).

In [ ]:
result = PilotScorer(config).audit()
result

## §1.2b — True tie-conversion via re-derived l2 rankings

`tie_conversion` above read 0 because it scored the persisted **v2** rankings, while the ties are
defined on the stronger **l2 (gemini)** stack whose ranked lists were never saved (only the scores
were). The gemini **collections still exist**, and the documents are already embedded in them — so
re-deriving the l2 rankings is **query-side only**: embed each pilot query with gemini, search, and
score before/after adding the judged gold. The query is embedded **server-side** by Qdrant's managed
inference, so this must hit the **cloud** instance that did the indexing: `QDRANT_CLOUD_URL` +
`QDRANT_CLOUD_API_KEY` + `cloud_inference=True` (same as `architecture_5k_test.ipynb`), and
`OPEN_ROUTER_API_KEY` for the embedding call.

In [ ]:
# Re-derive l2 (gemini) rankings from the live collections — query-side only (~$0.001, no re-index).
import os
from qdrant_client import QdrantClient

from scripts.legb import gemini_dense_cfg, SPARSE_CFG, LegBPilot
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
)
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.qrels import QrelStore
from relevance_judge.residual import above_gold, regime

# The gemini collections AND the managed inference live on the cloud instance — a plain
# QDRANT_URL has no InferenceService and errors with "InferenceService URL not configured".
QDRANT = bool(os.getenv("QDRANT_CLOUD_URL"))

if QDRANT:
    _client = QdrantClient(url=os.environ["QDRANT_CLOUD_URL"],
                           api_key=os.environ["QDRANT_CLOUD_API_KEY"],
                           timeout=120, cloud_inference=True)   # server-side query embedding
    _dense = gemini_dense_cfg()                                 # reads OPEN_ROUTER_API_KEY
    _pilot = LegBPilot(_client, _dense)                         # collection naming only
    _strats: dict[str, dict] = {}

    def l2_rankings(lane, query):
        coll = _pilot.collection(lane)
        if coll not in _strats:
            _strats[coll] = {
                "dense_only": DenseOnlyStrategy(_client, coll, _dense, SPARSE_CFG),
                "sparse_only": SparseOnlyStrategy(_client, coll, _dense, SPARSE_CFG),
                "pure_rrf": PureRRFStrategy(_client, coll, _dense, SPARSE_CFG),
            }
        return {r: st.rank(query) for r, st in _strats[coll].items()}

    print("ready — documents already embedded in the gemini collections; only queries are embedded")
else:
    print("set QDRANT_CLOUD_URL + QDRANT_CLOUD_API_KEY + OPEN_ROUTER_API_KEY to re-derive l2 rankings")

In [ ]:
# Sanity: the re-derived l2 dense/rrf score should reproduce the stored rows.json l2 (same rankings).
if QDRANT:
    import json
    draw = {(r["dataset"], str(r["query_id"])): r
            for r in json.loads((config.arch5k / "rows.json").read_text())}
    obj = RouterObjective(min_relevance=config.min_relevance)
    s = Sources(config)
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str})
    check = []
    for ds, qid in list(atoms.groupby(["dataset", "query_id"]).groups)[:6]:
        row = draw.get((ds, qid))
        if row is None:
            continue
        q = s.query_text(ds, {qid}).get(qid, "")
        gold = {d: 1 for d in s.manifest_gold(ds).get(qid, set())}
        redo = {r: obj.assess(rank, gold)[0] for r, rank in l2_rankings(ds, q).items()}
        check.append({"dataset": ds, "query_id": qid,
                      "stored_dense": round(row["l2"]["dense_only"], 3),
                      "redo_dense": round(redo["dense_only"], 3),
                      "stored_rrf": round(row["l2"]["pure_rrf"], 3),
                      "redo_rrf": round(redo["pure_rrf"], 3)})
    display(pd.DataFrame(check))
else:
    print("skipped (no QDRANT_URL)")

In [29]:
# True tie-conversion: re-retrieve l2 rankings for every judged row, score before (human gold)
# vs after (human + judged), and count all_tied rows that become decisive/low-margin.
# I/O-bound (server-side gemini embed + 3 Qdrant round-trips per row) — parallelized with
# 16 threads brings ~3h serial to ~12min. QdrantClient is thread-safe; the _strats cache
# init in l2_rankings has a benign race (same value written by concurrent threads).
if QDRANT:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm.auto import tqdm

    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    datasets = sorted(atoms["dataset"].unique())
    s = Sources(config)
    obj = RouterObjective(min_relevance=config.min_relevance)
    human = s.human_store(datasets)
    merged = QrelStore.concat([human, RelevanceJudge(config).as_qrelstore()])
    before = {ds: human.lookup(ds) for ds in datasets}
    after = {ds: merged.lookup(ds) for ds in datasets}

    def one(ds, qid):
        q = s.query_text(ds, {qid}).get(qid, "")
        if not q:
            return None
        rk = l2_rankings(ds, q)
        sb = {r: obj.assess(rank, before[ds].get(qid, {}))[0] for r, rank in rk.items()}
        sa = {r: obj.assess(rank, after[ds].get(qid, {}))[0] for r, rank in rk.items()}
        return {"dataset": ds, "query_id": qid,
                "regime_before": regime(sb), "regime_after": regime(sa),
                "resolved": regime(sb) == "all_tied" and regime(sa) != "all_tied"}

    work = list(atoms.groupby(["dataset", "query_id"]).groups)
    rows = []
    with ThreadPoolExecutor(max_workers=16) as ex:
        futures = [ex.submit(one, ds, qid) for ds, qid in work]
        for f in tqdm(as_completed(futures), total=len(futures), desc="l2 re-retrieve"):
            r = f.result()
            if r:
                rows.append(r)

    l2conv = pd.DataFrame(rows)
    tied = int((l2conv["regime_before"] == "all_tied").sum())
    broke = int(l2conv["resolved"].sum())
    print(f"rows: {len(l2conv)} | all_tied under l2 before: {tied} | broke after judged gold: {broke}")
    print("l2 tie_conversion (of all_tied rows):", round(broke / tied, 3) if tied else "n/a")
    display(l2conv.groupby(["regime_before", "regime_after"]).size().rename("rows"))
else:
    print("skipped (no QDRANT_URL)")


l2 re-retrieve:   0%|          | 0/300 [00:00<?, ?it/s]

rows: 300 | all_tied under l2 before: 300 | broke after judged gold: 63
l2 tie_conversion (of all_tied rows): 0.21


regime_before  regime_after
all_tied       all_tied        237
               low_margin       63
Name: rows, dtype: int64

## §1.2c — Corrected pilot: judge the l2 above-gold docs

The §1.2 pilot picked above-gold docs from the **v2** rankings, but the ties are **l2**-defined — so
a judged doc can sit outside l2's top-k and never move the l2 score. Here we re-derive each row's
above-gold set **from the l2 rankings**, judge the ones we missed, and re-measure conversion on the
right doc set. (Bonus: this also covers the 15 rows dropped earlier for gold-at-v2-rank-1.)

In [30]:
# Fetch l2 rankings once for every sub-1.0 tie, derive l2 above-gold, and diagnose the v2/l2 gap.
# Parallelized like §1.2b — same I/O-bound pattern, same 16-worker cap.
if QDRANT:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm.auto import tqdm
    s = Sources(config)
    obj = RouterObjective(min_relevance=config.min_relevance)
    sub1 = queue.residual()
    sub1 = sub1[(sub1["regime"] == "all_tied") & sub1["sub1"]].reset_index(drop=True)
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    judged = set(zip(atoms["dataset"], atoms["query_id"], atoms["doc_id"]))

    def one(row):
        gold = s.manifest_gold(row.dataset).get(row.query_id, set())
        rk = l2_rankings(row.dataset, row.query)
        above = above_gold(
            {route: obj.ordered(scores) for route, scores in rk.items()}, gold
        )
        return (row.dataset, row.query_id, rk, above)

    l2_cache, l2_above = {}, {}
    with ThreadPoolExecutor(max_workers=16) as ex:
        futures = [ex.submit(one, r) for r in sub1.itertuples(index=False)]
        for f in tqdm(as_completed(futures), total=len(futures), desc="l2 above-gold"):
            ds, qid, rk, above = f.result()
            l2_cache[(ds, qid)] = rk
            l2_above[(ds, qid)] = above

    total_above = sum(len(a) for a in l2_above.values())
    already = sum(1 for (k, a) in l2_above.items() for d in a if (k[0], k[1], d) in judged)
    judged_in_l2 = sum(
        1 for (ds, qid, did) in judged
        if did in set().union(*(set(o) for o in l2_cache.get((ds, qid), {}).values()), set())
    )
    print(f"sub-1.0 ties: {len(sub1)} | l2 above-gold docs: {total_above} "
          f"(already judged: {already}, MISSED: {total_above - already})")
    print(f"of {len(judged)} judged pairs, only {judged_in_l2} land in some l2 top-k "
          f"-> the rest could not have moved the l2 score (the v2/l2 mismatch, quantified)")
else:
    print("skipped (no QDRANT_CLOUD_URL)")


l2 above-gold:   0%|          | 0/85 [00:00<?, ?it/s]

sub-1.0 ties: 85 | l2 above-gold docs: 141 (already judged: 10, MISSED: 131)
of 4816 judged pairs, only 181 land in some l2 top-k -> the rest could not have moved the l2 score (the v2/l2 mismatch, quantified)


In [31]:
# Build the pairs we MISSED: l2 above-gold docs not yet judged.
if QDRANT:
    rows = []
    qtext = dict(zip(zip(sub1["dataset"], sub1["query_id"]), sub1["query"]))
    for (ds, qid), above in l2_above.items():
        todo = {d for d in above if (ds, qid, d) not in judged}
        if not todo:
            continue
        texts = s.corpus_text(ds, todo)
        for doc_id in todo:
            text = texts.get(doc_id)
            if text:
                rows.append({"dataset": ds, "query_id": qid, "doc_id": doc_id,
                             "query": qtext[(ds, qid)], "doc_text": text})
    new_pairs = pd.DataFrame(rows)
    print(f"new l2-above-gold pairs to judge: {len(new_pairs)}")
    new_pairs.head()
else:
    print("skipped")

new l2-above-gold pairs to judge: 131


In [32]:
# Judge the missed pairs (idempotent — skips the 371 already banked). Needs SPEND + a passed gate.
if QDRANT and SPEND and len(new_pairs):
    runs = JudgeRunLog(config).load()
    passed = runs[runs["passed"]] if not runs.empty else runs
    run_id = str(passed.iloc[-1]["judge_run_id"]) if not passed.empty else None
    counts = (RelevanceJudge(config).judge_pairs(new_pairs, run_id=run_id, budget=budget())
              if run_id else {"error": "no passed validation run"})
    counts
elif QDRANT:
    print(f"set SPEND=True to judge {len(new_pairs)} missed pairs (~a few cents)")
else:
    print("skipped")

relevance:   0%|          | 0/131 [00:00<?, ?pair/s]

  131 hops, 192,478 tokens, 19.0s wall (187.8s of it LLM) -> 1.0 hops/row, 1,469 tokens/row, 0.1s/row


In [33]:
# TRUE tie-conversion: l2 rankings (cached) scored before (human) vs after (human + ALL judged now).
if QDRANT:
    atoms = RelevanceJudge(config).load().astype({"query_id": str, "doc_id": str, "relevance": int})
    datasets = sorted(atoms["dataset"].unique())
    human = s.human_store(datasets)
    merged = QrelStore.concat([human, RelevanceJudge(config).as_qrelstore()])
    obj = RouterObjective(min_relevance=config.min_relevance)

    conv = []
    for (ds, qid), rk in l2_cache.items():
        gb = human.lookup(ds).get(qid, {})
        ga = merged.lookup(ds).get(qid, {})
        sb = {r: obj.assess(rank, gb)[0] for r, rank in rk.items()}
        sa = {r: obj.assess(rank, ga)[0] for r, rank in rk.items()}
        conv.append({"regime_before": regime(sb), "regime_after": regime(sa),
                     "resolved": regime(sb) == "all_tied" and regime(sa) != "all_tied"})
    conv = pd.DataFrame(conv)
    tied = int((conv["regime_before"] == "all_tied").sum())
    broke = int(conv["resolved"].sum())
    print(f"rows: {len(conv)} | all_tied before: {tied} | broke after judged gold: {broke}")
    print("TRUE l2 tie_conversion (l2 above-gold docs judged):",
          round(broke / tied, 3) if tied else "n/a")
    display(conv.groupby(["regime_before", "regime_after"]).size().rename("rows"))
else:
    print("skipped")

rows: 85 | all_tied before: 79 | broke after judged gold: 12
TRUE l2 tie_conversion (l2 above-gold docs judged): 0.152


regime_before  regime_after   
all_tied       all_tied           67
               decisive_strong     1
               low_margin         11
low_margin     all_tied            1
               decisive_strong     1
               low_margin          4
Name: rows, dtype: int64

## §1.3 — Eye test: what each stage actually produces

Every number above is an aggregate. This section prints the real
(query, document, verdict) triples behind them, because a precision figure is
only worth what the labels under it are worth.

**Where the "human" comes from.** `human_relevant` is not ours — it is the
relevance grade that shipped with the lane's `qrels.parquet` (BEIR, MIRACL,
freshstack annotators). We only sample it. That is what makes accuracy
measurable in stage 1, and why it can only be measured on lanes whose
annotators also marked documents **irrelevant**.

| | stage 1 `judge_rows()` | stage 2 `judge_pairs()` |
|---|---|---|
| artifact | `validation_predictions.parquet` | `judged_qrels.parquet` |
| population | pairs a human **already judged** | pairs **nobody** judged |
| `human_relevant` | yes — the whole point | absent by construction |
| output | `pred` vs `human` → precision | `relevance` → **permanent gold** |
| purpose | **measure** the instrument | **use** the instrument |
| idempotent | no | yes, on `(dataset, query_id, doc_id)` |

They are chained by the gate: **stage 1 measures → gate passes → stage 2
spends**, and every atom carries the `judge_run_id` of the gate that authorized
it, so any gold label traces back to a measured precision.

**What to look for.** In stage 1, the disagreements are the interesting rows —
some are judge errors, and some are the judge being *stricter than the
annotator*. In stage 2 there is nothing to compare against, so the judge's
`reason` (and, for atoms banked after 2026-09-04, its `evidence` quote) is the
only thing standing between a plausible label and permanent wrong gold.

In [34]:
# Stage 1 — the judge against a human verdict. Free: reads only what is on disk.
from scripts.judge_eye_test import stage_one, stage_two

stage_one(config, Sources(config), per_case=2)

STAGE 1  judge_rows()  ->  validation_predictions.parquet
The HUMAN column is not ours: it is the relevance grade that SHIPPED with
the lane's qrels.parquet (BEIR / MIRACL / freshstack annotators). We only
sample it. That is what makes accuracy measurable here — and why it can
only be measured on lanes whose annotators also marked docs IRRELEVANT.


--- AGREE, relevant     (true positive)   [759 rows] ---

  [beir-touche-2020  q=11  doc=21fa6aa9-2019-04-18T15:26:34Z-00003-000]
  QUERY     Should performance-enhancing drugs be accepted in sports?
  DOC       Performance Enhancement Drugs in Professional Sports Performance Enhancements are Putting Athletes
            in Danger Athletes are cheating more and more now by taking performance enhancements for
            their professional sport. An athletes health could be at risk while taking performance
            enhancement drugs, we alr
  HUMAN     RELEVANT   (qrels grade 2)
  JUDGE     RELEVANT   — the document explicitly argues PEDs

In [35]:
# Stage 2 — the gold it invents where no human verdict exists.
stage_two(config, Sources(config), per_case=2)


STAGE 2  judge_pairs()  ->  judged_qrels.parquet
No HUMAN column exists here, by construction: these are the pairs nobody
judged. The judge's verdict BECOMES the gold, which is why stage 1 has to
pass a precision gate before this is allowed to run at all.


--- QRELS HOLE FILLED  (judged relevant -> new gold)   [184 atoms] ---

  [clerc  q=756700  doc=22657509]
  QUERY     the district court is debatable or wrong and that any dispositive procedural ruling by the district
            court is likewise debatable. Miller-El v. Cockrell, 537 U.S. 322, 336-38, 123 S.Ct.
            1029, 154 L.Ed.2d 931 (2003); Slack v. McDaniel, 529 U.S. 473, 484, 120 S.Ct. 1595, 146
            L.Ed.2d 542 (2000); Rose v.
  DOC       satisfy § 2253(c) is straightforward: The petitioner must demonstrate that reasonable jurists would
            find the district court’s assessment of the constitutional claims debatable or wrong.
            The issue becomes somewhat more complicated where, as here, the d

In [ ]:
# The two artifacts share (dataset, query_id, doc_id) but are near-DISJOINT by
# design: stage 1 needs a human label, stage 2 needs its absence. Overlap ~= 0
# is the invariant; anything else means the sampler reached into judged pairs.
import pandas as pd

_v = pd.read_parquet(config.validation_predictions)
_a = RelevanceJudge(config).load()
_k = ["dataset", "query_id", "doc_id"]
print(f"stage 1 rows: {len(_v):,}   lanes: {_v.dataset.nunique()}")
print(f"stage 2 atoms: {len(_a):,}   lanes: {_a.dataset.nunique()}")
print(f"overlap (expect ~0): {len(_v[_k].astype(str).merge(_a[_k].astype(str)))}")
print()
print("atoms per authorizing gate run:")
print(_a.judge_run_id.value_counts().to_string())

## Next steps

- **Not built (gated):** the §3a formal 500–1K sample-test router-population arm (placebo +
  ≥5 seeds) and Phase 2 dataset-wide deepening (~72K rows, ~$390) — both wait on the SPEC
  decision *"may the route label be defined below current truth depth?"*.
- **Calibration:** luna prices in `config.py` are placeholders — set them from luna's real card.
- **CLI twin** for headless runs: `poetry run python src/scripts/run_relevance_judge.py --audit | --validate | --pilot-sub1 | --score`.

# Conclusion — what the atoms bought (2026-09-03)

**The question**: can judging tail documents break the ties measurement left
unlabelled, and is the result worth anything to the router?

**Answered: yes it breaks them, and no it does not help the router.** Both
halves are load-bearing.

## The measured result

5,133 atoms over 354 rows. Rescoring the persisted v2 rankings under
`human ∪ judged` gold:

| before ＼ after | all_tied | low_margin | decisive_strong |
|---|---|---|---|
| **all_tied** | 189 | **71** | **0** |
| **low_margin** | 1 | 88 | 5 |

- **71 of 260 tied rows (27.3%) broke.** Ties are *not* encoder-invariant. The
  NDCG lever is real, and 0 of 1,595 perfect ties were structurally unbreakable
  (SPEC d67).
- **0 of those 71 reached `decisive_strong`.** Every break landed in
  `low_margin`.
- **`qrels_hole_rate_rows` = 35.6%** — a third of judged rows had a relevant
  document missing from qrels. That is the real deliverable.

Note the published `tie_conversion_rate` (0.209) *understates* the effect: its
denominator is all 354 rescored rows, including the 94 that were never tied.

## The true-l2 run — and why 2 rows DID reach decisive

The table above rescores **v2** rankings, because l2's ranked lists were never
persisted. Re-retrieving l2 live (§1.2c, Qdrant) over the sub-1.0 tie
population — all 85 of them — closes that caveat and gives a different answer:

```
rows: 85 | all_tied before: 80 | broke after judged gold: 18
TRUE l2 tie_conversion: 0.225

all_tied  -> all_tied          62
          -> low_margin        16
          -> decisive_strong    2     <-- non-zero
low_margin -> low_margin         5
```

That is **not** a contradiction. The two runs cover complementary halves of the
tie mass, and the mechanism predicts both:

| population | share of `all_tied` | tied | broke | → decisive | → low_margin |
|---|---|---|---|---|---|
| **perfect** (score = 1.0) | 1,832 (95.6%) | 260 (v2) | 71 (27.3%) | **0** | 71 |
| **sub-1.0** (score < 1.0) | 85 (4.4%) | 80 (l2) | 18 (22.5%) | **2** | 16 |

A **perfect** tie has gold at rank 1 in *every* route, so `HitRate@1` is already
maxed and only the `0.3·NDCG` term can move — capped below the router's
`hit_weight − ndcg_weight` = 0.4 bar. Hence 0.

A **sub-1.0** tie does *not* have gold at rank 1, so judging an **above-gold**
document relevant flips `HitRate@1` and the margin can clear 0.4. Hence 2.

So d67(d)'s ceiling holds exactly where it was argued — on perfect ties — and the
`above_gold` lever that d67(b) deliberately kept is what produces the only
decisive rows the program yields.

**Router yield is still negligible: 2 decisive rows out of 1,917 tied rows
(0.1%)**, or roughly 26 rows scaled to v2-100K's 24,751 ties. The program remains
**qrels depth / dataset supply, and nothing else**. Any ledger that reads these
conversions as router gain is wrong.

**Sparse finds more missing gold than dense** — discovered-relevance 6.3% sparse
/ 6.5% rrf / 5.0% dense. A small counter-signal to arch5k finding 4's 87%-dense
skew, pointing the same way as the os_distill sparse upgrade. Not a router claim.

## How the judge got fast AND accurate

luna is a reasoning model, and the latency was entirely hidden reasoning:

| configuration | latency | precision |
|---|---|---|
| full reasoning, verdict-first prompt | ~60 s | 0.955 |
| suppressed, verdict-first prompt | **1.2 s** | 0.905 |
| suppressed, `ASKED:` evidence line first | 1.2 s | 0.932 |
| suppressed, instruction rewritten from the failure brief | **1.2 s** | **0.988** |

Two lessons. Sending `reasoning_effort` as the **top-level** param (not only
nested in `extra_body`) is what suppresses it — 50× faster. And **verdict-first
is specifically wrong for a non-reasoning judge**: it commits before examining,
then rationalises. Forcing a short evidence field *before* the verdict recovers
most of the loss for free, and a rewrite driven by the measured failures
(`docs/research/judge-prompt-brief.md`) beat full reasoning outright.

## Caveats that travel with these numbers

- **Precision 0.988 is a transfer estimate.** The 12 negative-bearing referee
  lanes and the 6 lanes the judge is applied to are **disjoint** — deploy lanes
  ship positive-only qrels, so precision is structurally unmeasurable there. The
  pseudo-negative check bounds gross over-calling on the deploy lanes; the
  near-miss boundary, where a relevance judge actually fails, stays unmeasured
  where it runs.
- **Stack mismatch — RESOLVED for the sub-1.0 population.** `PilotScorer`'s
  27.3% rescores v2 rankings under an l2-selected population; §1.2c now measures
  the sub-1.0 ties on live l2 rankings (22.5%, 2 decisive). The 95.6% perfect-tie
  majority is still v2-rescored only — its l2 lists were never persisted, and
  re-retrieving all 1,832 would be a fresh Qdrant pass.
- **Provenance spans two judge configurations**: 371 atoms from the 09-01 run
  (0.955, old prompt, full reasoning), 4,762 from 09-03 (0.988, new prompt,
  suppressed). Both gates passed; the ledger is still two instruments.
- **`reasoning_effort` middle untested.** 'none' and full reasoning were never
  compared under the *same* prompt, so "suppression costs precision" is
  confounded with "the old prompt was worse".
- **No working per-call timeout.** Neither litellm's `timeout=` kwarg nor the
  module global bounds a call on this path (measured: a `timeout=30` call
  completed at 60.9 s). One stalled request parks a worker indefinitely.

## Four paid-path defects fixed along the way

Each has a regression test that fails against the old code:

1. `judge_pairs` / `judge_rows` flushed their buffer **after** the loop, so a
   budget stop discarded up to 99 (resp. 49) already-**paid** verdicts.
2. `judge_one` caught only `TRANSIENT_PROVIDER_ERRORS` — any other provider
   error abandoned every remaining pair (observed: a run died after 2 of 3,600).
3. A reply cut off at the token cap loses its `VERDICT:` line and parses to
   nothing: **249 paid pairs (5.2%) lost in one run.** Now `finish_reason ==
   "length"` triggers one retry at double the cap; the rerun recovered all 249
   with **0 unreadable**.
4. Swallowed errors were invisible, which is why a 60 s/call provider looked like
   a hang and a format regression looked like a provider blip. Cause counts, first
   message per type, and raw unparseable samples are now printed by both paths.

Cost is also now recorded (`spend_usd` in the report, live on the progress bar),
and the engine rates were **2× too high** — corrected to the published
$0.20/$1.20, which re-costs the full tie run from $13.10 to ~$7.00.

## Next

**Step 2 gates everything downstream**: with |gold| ≥ 2, what is the margin
distribution, and does the enriched gold change the router's decisive population
at all? A second gold doc *can* flip rank-1 — unlike the NDCG-only path — so that
is the only place decisive rows could come from. Tracked in TODOS under SPEC d68.